# Content two-tower MLP: candidate-generation baseline

A personalised candidate generator trained on the full `labels_train` pool,
with up to four exposed negatives redrawn per positive each epoch.
Each request's causal history is mean-pooled over valid text and entity
vectors and encoded by a user tower. Each article is encoded by a separate
item tower from its long-term intrinsic content only: text and entity vectors
plus missing-vector flags. There are no popularity, user-ID or article-ID
inputs in the item tower. The score is a scaled normalised dot product.

Shared protocol:

- Use `full_history_rows` from the request profile for that impression. Never
  reuse a later request's profile for an earlier request.
- Exclude every article in the request's full causal history.
- Eligibility is `available_from <= timestamp`.
- Train on the full `labels_train` rows, tune chronologically on the tail of
  train, then refit on all training rows and evaluate complete validation
  slates.
- Popularity is not a model feature. The rolling prior-hour click order is used
  only for tie-breaking and cold or short-list fallback. A request in hour `h`
  only uses the completed hour `h - 1`, matching the rolling-popularity
  baseline. It is strictly causal per request hour, not frozen split-wide
  aggregates.
- The score is a ranking score. Training uses grouped exposed negatives, so it
  is not a calibrated click probability.

The notebook runs the full pipeline on the complete training and target splits.


## 1. Setup

In [1]:
from __future__ import annotations

import json
import sys
import time
from pathlib import Path

import numpy as np
import polars as pl
import torch
import torch.nn.functional as F
from torch import nn

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "run_all.py").is_file()
)
CG_DIR = ROOT / "5_Models" / "5_01_Candidate_Generation"
sys.path.insert(0, str(CG_DIR))
sys.path.insert(0, str(ROOT / "src"))

from cf_models import resolve_device
from cg_result_writer import CandidateWriter
from neural_helpers import (
    EarlyStopper,
    EpochNegativeSampler,
    chronological_cutoff,
    chronological_mask,
    max_clicks_per_impression,
    mean_slate_metrics,
    tuning_slates,
)
from newsrec.common import Settings

PROCESSED = ROOT / "1_Data_Preparation" / "processed"
FEATURES = ROOT / "4_Feature_Engineering" / "artifacts" / "prequential-all"
REPRESENTATIONS = ROOT / "3_Representations" / "artifacts"
ARTIFACTS = ROOT / "5_Models" / "5_01_Candidate_Generation" / "artifacts"
RESULTS = ROOT / "results"

TARGET_SPLIT = "validation"
FIT_SPLIT = "train"
BASE_MODEL_NAME = "content_two_tower_mlp"
RUN_LOSSES = ("bpr",)
LOSS_CONFIGS = {}
for loss in RUN_LOSSES:
    artifact_dir = ARTIFACTS / f"two_tower_content_{loss}"
    artifact_dir.mkdir(parents=True, exist_ok=True)
    LOSS_CONFIGS[loss] = {
        "model_name": f"{BASE_MODEL_NAME}_{loss}",
        "artifact_dir": artifact_dir,
        "recommendations_path": artifact_dir
        / f"recommendations_{TARGET_SPLIT}_top500.parquet",
        "result_path": RESULTS
        / f"candidate_generation_{BASE_MODEL_NAME}_{loss}_{TARGET_SPLIT}.csv",
    }
PROFILE_DIR = ARTIFACTS / "two_tower_content_bpr"
PROFILE_DIR.mkdir(parents=True, exist_ok=True)
COMPARISON_PATH = RESULTS / f"candidate_generation_comparison_{TARGET_SPLIT}.csv"

M_VALUES = (50, 100, 150, 200, 250, 300, 350, 400, 450, 500, 550, 600, 650, 700, 750)
CANDIDATE_CAP = max(M_VALUES)
TEXT_DIM = 64
ENTITY_DIM = 100
FEATURE_DIM = TEXT_DIM + ENTITY_DIM + 2
HIDDEN_DIM = 256
EMBED_DIM = 128
DROPOUT = 0.1
NUM_EPOCHS = 50
BATCH_SIZE = 16_384
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5
SEED = 4222
PATIENCE = 7
TUNE_QUANTILE = 0.9
TUNE_CANDIDATES = PROCESSED / "candidates_train.parquet"
DEVICE = resolve_device("auto")

HOUR_NS = 3_600 * 10**9
POPULARITY_SPLITS = ("train", "validation")

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Device:", DEVICE)


Device: cuda


## 2. Data ingestion and model-related processing

### History pooling and feature builders

`pool_profiles` mean-pools valid article vectors and L2-normalises each block.
`profile_features` builds the 166-dimensional request input. `content_features`
builds the 166-dimensional intrinsic article-content block, which is the only
input to the item tower.

In [2]:
def pool_profiles(history_rows, matrix, block=2_000):
    matrix = np.asarray(matrix)
    n = len(history_rows)
    out = np.zeros((n, matrix.shape[1]), dtype=np.float32)
    for start in range(0, n, block):
        chunk = history_rows[start : start + block]
        lengths = np.fromiter(
            (len(rows) for rows in chunk), dtype=np.int64, count=len(chunk)
        )
        if lengths.sum() == 0:
            continue
        flat = np.concatenate(
            [np.asarray(rows, dtype=np.int64) for rows in chunk if len(rows)]
        )
        vectors = np.asarray(matrix[flat], dtype=np.float32)
        valid = np.linalg.norm(vectors, axis=1) > 1e-12
        if not valid.any():
            continue
        groups = np.repeat(np.arange(len(chunk), dtype=np.int64), lengths)[valid]
        vectors = vectors[valid]
        starts = np.r_[0, np.flatnonzero(groups[1:] != groups[:-1]) + 1]
        pooled = np.add.reduceat(vectors, starts, axis=0) / np.diff(
            np.r_[starts, len(groups)]
        )[:, None]
        norms = np.linalg.norm(pooled, axis=1)
        pooled /= np.where(norms > 0, norms, 1)[:, None]
        out[start : start + len(chunk)][groups[starts]] = pooled
    return out


In [3]:
def profile_features(history_rows):
    text_profile = pool_profiles(history_rows, TEXT)
    entity_profile = pool_profiles(history_rows, ENTITY)
    out = np.zeros((len(history_rows), FEATURE_DIM), dtype=np.float32)
    out[:, :TEXT_DIM] = text_profile
    out[:, TEXT_DIM : TEXT_DIM + ENTITY_DIM] = entity_profile
    out[:, -2] = np.linalg.norm(text_profile, axis=1) > 0
    out[:, -1] = np.linalg.norm(entity_profile, axis=1) > 0
    return out


In [4]:
def content_features(text, entity):
    out = np.zeros((text.shape[0], FEATURE_DIM), dtype=np.float32)
    out[:, :TEXT_DIM] = text
    out[:, TEXT_DIM : TEXT_DIM + ENTITY_DIM] = entity
    out[:, -2] = np.linalg.norm(text, axis=1) > 0
    out[:, -1] = np.linalg.norm(entity, axis=1) > 0
    return out


### Causal request-profile caches

The profile row for each impression contains the articles clicked before that
request. The cache stores one 166-dimensional input per request: `text64`
profile, `entity100` profile and two availability flags. Repeated history
references are preserved, zero vectors are ignored and each block is
L2-normalised. The profile index also supplies the
`impression_uid -> profile_row` map used to join training rows.

In [5]:
def load_profiles(split):
    parts = sorted((FEATURES / f"request_profiles_{split}").glob("*.parquet"))
    return pl.read_parquet(
        parts, columns=["impression_uid", "timestamp", "full_history_rows"]
    ).sort("timestamp", maintain_order=True)


In [6]:
def profile_cache_paths(split):
    return (
        PROFILE_DIR / f"user_features_{split}.npy",
        PROFILE_DIR / f"profile_uids_{split}.npy",
        PROFILE_DIR / f"profile_index_{split}.parquet",
    )


In [7]:
def load_profile_cache(split):
    feature_path, uid_path, index_path = profile_cache_paths(split)
    if feature_path.exists() and uid_path.exists() and index_path.exists():
        return np.load(feature_path, mmap_mode="r"), np.load(uid_path)

    frame = load_profiles(split)
    n = frame.height
    features = np.lib.format.open_memmap(
        feature_path, mode="w+", dtype=np.float32, shape=(n, FEATURE_DIM)
    )
    features[:] = profile_features(frame["full_history_rows"].to_list())
    features.flush()
    uids = frame["impression_uid"].to_numpy()
    np.save(uid_path, uids)
    pl.DataFrame(
        {"impression_uid": uids, "profile_row": np.arange(n, dtype=np.int32)}
    ).write_parquet(index_path)
    print(f"Built profile cache for {split}: {n:,} requests")
    return np.load(feature_path, mmap_mode="r"), uids


In [8]:
TEXT = np.load(REPRESENTATIONS / "text64.npy", mmap_mode="r")
ENTITY = np.load(REPRESENTATIONS / "article_entity100.npy", mmap_mode="r")

TRAIN_USER_FEATURES, _ = load_profile_cache(FIT_SPLIT)
PROFILE_INDEX_TRAIN = pl.read_parquet(profile_cache_paths(FIT_SPLIT)[2])

print(f"Train user feature shape: {TRAIN_USER_FEATURES.shape}")


Built profile cache for train: 95,071 requests
Train user feature shape: (95071, 166)


### Article content inputs and the time-eligible catalogue

Article vectors are indexed by `article_row`, with row 0 reserved as padding.
The item tower receives only intrinsic content vectors and missing-vector flags.
Rolling popularity is not an item-tower feature; it is used only for candidate
tie-breaking and the empty-history fallback. The catalogue is ordered by
`available_from` so each request timestamp selects a prefix.

In [9]:
article_index = pl.read_parquet(
    REPRESENTATIONS / "article_index.parquet", columns=["article_row", "news_code"]
).sort("article_row")
n_items = article_index.height + 1

availability = pl.read_parquet(
    REPRESENTATIONS / "article_availability.parquet",
    columns=["news_code", "available_from"],
)
catalogue = article_index.join(availability, on="news_code", how="left")
ordered = catalogue.sort(["available_from", "news_code"])
ARTICLE_ROWS_SORTED = ordered["article_row"].to_numpy().astype(np.int64)
NEWS_SORTED = ordered["news_code"].to_numpy().astype(np.int64)
AVAILABLE_SORTED = ordered["available_from"].to_numpy().astype("datetime64[ns]")

CONTENT_INPUTS = content_features(TEXT, ENTITY)
CONTENT_INPUTS_SORTED = CONTENT_INPUTS[ARTICLE_ROWS_SORTED]
POS_IN_SORTED = np.full(n_items, -1, dtype=np.int64)
POS_IN_SORTED[ARTICLE_ROWS_SORTED] = np.arange(len(ARTICLE_ROWS_SORTED))

print(f"Catalogue articles: {article_index.height:,}")


Catalogue articles: 51,282


### Rolling prior-hour click order

Click counts are aggregated by `(news_code, hour)` from the logged candidate
slates. A request in hour `h` uses only the previous completed hour `h - 1`,
so the ordering is strictly
causal. It is used only for candidate tie-breaking and the empty-history
fallback; it is not an input to the item tower. The source splits and hour grid
match `CG_rolling_popularity_baseline.ipynb`.

In [10]:
index_codes = article_index["news_code"].to_numpy().astype(np.int64)
row_of_code = np.zeros(int(index_codes.max()) + 1, dtype=np.int64)
row_of_code[index_codes] = article_index["article_row"].to_numpy().astype(np.int64)

hourly = (
    pl.concat(
        [
            pl.scan_parquet(PROCESSED / f"candidates_{split}.parquet").select(
                "news_code",
                "clicked",
                (pl.col("timestamp").dt.epoch("ns") // HOUR_NS)
                .cast(pl.Int64)
                .alias("hour"),
            )
            for split in POPULARITY_SPLITS
        ]
    )
    .group_by("news_code", "hour")
    .agg(clk=pl.col("clicked").cast(pl.Int64).sum())
    .collect()
)

source_rows = row_of_code[hourly["news_code"].to_numpy().astype(np.int64)]
MIN_HOUR = int(hourly["hour"].min())
MAX_HOUR = int(hourly["hour"].max())
N_HOURS = MAX_HOUR - MIN_HOUR + 1
CLICKS = np.zeros((N_HOURS, n_items), dtype=np.int32)
hour_positions = hourly["hour"].to_numpy().astype(np.int64) - MIN_HOUR
CLICKS[hour_positions, source_rows] = hourly["clk"].to_numpy().astype(np.int32)


def hour_index_of(timestamps_ns):
    return np.asarray(timestamps_ns, dtype=np.int64) // HOUR_NS - MIN_HOUR


def hourly_popularity(hour_index, item_rows):
    return np.log1p(CLICKS[hour_index - 1, item_rows])


print(f"Rolling popularity hours: {N_HOURS:,}")
print(f"Articles clicked at least once: {int((CLICKS.sum(0) > 0).sum()):,}")


Rolling popularity hours: 120
Articles clicked at least once: 6,398


### Training groups and chronological tuning slice

Training rows come from the full `labels_train` pool: each clicked positive is
paired with up to four exposed non-clicks redrawn from the same request each
epoch. The last 10% of training timestamps are held out for early stopping; the tuning slates come
from the complete `candidates_train.parquet`. Popularity is not attached to
training or tuning rows; it is used only at retrieval for ties and fallback.

In [11]:
run_manifest = json.loads((FEATURES / "run_manifest.json").read_text())
if run_manifest.get("training_candidate_pool") != "complete":
    raise ValueError(
        "Per-epoch sampling needs the complete training pool. "
        "Rebuild stage 4 with the default training_negative_sampling='epoch'."
    )
CFG = Settings(**run_manifest["config"])

TRAINING_ROWS = (
    pl.read_parquet(sorted((FEATURES / "labels_train").glob("*.parquet")))
    .select("impression_uid", "news_code", "clicked")
    .to_pandas()
)
if TRAINING_ROWS["clicked"].isna().any() or not TRAINING_ROWS[
    "clicked"
].isin([0, 1]).all():
    raise ValueError("Training rows must be uniquely binary-labelled")

train_profile_index = pl.read_parquet(
    sorted((FEATURES / f"request_profiles_{FIT_SPLIT}").glob("*.parquet")),
    columns=["impression_uid", "timestamp"],
)
profile_lookup = PROFILE_INDEX_TRAIN.join(
    train_profile_index, on="impression_uid", how="left"
)

positive_uids = np.unique(
    TRAINING_ROWS.loc[TRAINING_ROWS["clicked"].eq(1), "impression_uid"].to_numpy()
)
positive_profiles = profile_lookup.filter(
    pl.col("impression_uid").is_in(positive_uids)
)
cutoff_ns = chronological_cutoff(
    positive_profiles["timestamp"].to_numpy(), TUNE_QUANTILE
)
fit_profiles = chronological_mask(profile_lookup["timestamp"].to_numpy(), cutoff_ns)
fit_uids = set(
    profile_lookup.filter(pl.Series("fit", fit_profiles))["impression_uid"].to_list()
)
FIT_ROWS = TRAINING_ROWS[
    TRAINING_ROWS["impression_uid"].isin(fit_uids)
].reset_index(drop=True)
ALL_ROWS = TRAINING_ROWS
n_groups = int(TRAINING_ROWS["clicked"].sum())
EPOCH_SAMPLER = EpochNegativeSampler(ALL_ROWS, CFG)


def sample_epoch(epoch, *, fit_only):
    """Draw exposed negatives for one epoch and build content-MLP arrays."""
    groups, _ = EPOCH_SAMPLER.draw(
        int(epoch), uids=fit_uids if fit_only else None
    )
    groups = (
        pl.from_pandas(groups)
        .join(profile_lookup, on="impression_uid", how="left")
        .with_row_index("group_row")
        .join(
            article_index.select(
                pl.col("news_code"),
                pl.col("article_row").alias("pos_row"),
            ),
            left_on="positive_news_code",
            right_on="news_code",
            how="left",
        )
    )
    if groups["profile_row"].null_count() or groups["pos_row"].null_count():
        raise ValueError("Sampled groups are missing training profiles or article rows")
    negative_long = (
        groups.select("group_row", "negative_news_codes")
        .explode("negative_news_codes")
        .rename({"negative_news_codes": "negative_news_code"})
        .drop_nulls("negative_news_code")
        .with_columns(pl.int_range(pl.len()).over("group_row").alias("neg_pos"))
        .join(
            article_index.select(
                pl.col("news_code").alias("negative_news_code"),
                pl.col("article_row").alias("neg_row"),
            ),
            on="negative_news_code",
            how="left",
        )
        .sort(["group_row", "neg_pos"])
    )
    if negative_long["neg_row"].null_count():
        raise ValueError("Sampled negatives are missing article rows")
    n_epoch_groups = groups.height
    NEG_MAX = int(groups["negative_news_codes"].list.len().max() or 0)
    if n_epoch_groups and NEG_MAX == 0:
        raise ValueError("Sampled training rows contain no exposed negatives")
    NEG_MATRIX = np.zeros((n_epoch_groups, NEG_MAX), dtype=np.int64)
    group_rows = negative_long["group_row"].to_numpy().astype(np.int64)
    NEG_MATRIX[
        group_rows, negative_long["neg_pos"].to_numpy().astype(np.int64)
    ] = negative_long["neg_row"].to_numpy().astype(np.int64)
    NEG_LEN = np.bincount(group_rows, minlength=n_epoch_groups).astype(np.int64)
    return (
        groups["profile_row"].to_numpy().astype(np.int32),
        groups["pos_row"].to_numpy().astype(np.int32),
        NEG_MATRIX,
        NEG_LEN,
    )


TUNE_NDCG_K = max_clicks_per_impression(TUNE_CANDIDATES)

tune = tuning_slates(
    TUNE_CANDIDATES,
    cutoff_ns,
    article_index,
    PROFILE_INDEX_TRAIN,
    max_impressions=None,
    seed=SEED + 1,
)
TUNE_PROFILE_ROWS = tune["profile_row"].to_numpy().astype(np.int32)
TUNE_ITEM_ROWS = tune["article_row"].to_numpy().astype(np.int32)
TUNE_CLICKED = tune["clicked"].to_numpy().astype(np.float32)
TUNE_UIDS = tune["impression_uid"].to_numpy()
print(f"Training rows: {len(TRAINING_ROWS):,}")
print(f"Training positives: {n_groups:,}")
print(f"Chronological fit rows: {len(FIT_ROWS):,} / {len(ALL_ROWS):,}")
print(f"Sampler negatives per positive: up to {CFG.negative_count}")
print("Tuning cutoff:", np.datetime64(cutoff_ns, "ns"))
print(f"Tuning impressions: {tune['impression_uid'].n_unique():,}")
print(f"Tuning pairs: {tune.height:,}")


Training rows: 3,389,342
Training positives: 141,558
Chronological fit rows: 3,046,331 / 3,389,342
Sampler negatives per positive: up to 4
Tuning cutoff: 2019-11-12T13:50:10.000000000
Tuning impressions: 9,507
Tuning pairs: 343,011


## 3. Setting up the model

### Two-tower network definition

`TwoTower` has two independent MLPs. The request tower consumes the pooled
166-dimensional history input; the item tower consumes the 166-dimensional
intrinsic content input. Both outputs are L2-normalised and combined by a
scaled dot product. There are no popularity, user-ID or article-ID embeddings.

In [12]:
class TwoTower(nn.Module):
    def __init__(
        self,
        input_dim=FEATURE_DIM,
        hidden_dim=HIDDEN_DIM,
        embed_dim=EMBED_DIM,
        dropout=DROPOUT,
        scale=10.0,
    ):
        super().__init__()
        self.user_tower = self._tower(input_dim, hidden_dim, embed_dim, dropout)
        self.item_tower = self._tower(input_dim, hidden_dim, embed_dim, dropout)
        self.logit_scale = nn.Parameter(
            torch.tensor(float(np.log(scale)), dtype=torch.float32)
        )

    @staticmethod
    def _tower(input_dim, hidden_dim, embed_dim, dropout):
        return nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, embed_dim),
        )

    def encode_users(self, values):
        return F.normalize(self.user_tower(values), dim=-1)

    def encode_items(self, values):
        return F.normalize(self.item_tower(values), dim=-1)

    def forward(self, user_values, item_values):
        return torch.exp(self.logit_scale) * (
            self.encode_users(user_values) * self.encode_items(item_values)
        ).sum(dim=-1)


## 4. Training the model

### Two-tower training, chronological tuning and refit

The user tower consumes the 166-dimensional pooled history input. The item tower
consumes the 166-dimensional intrinsic content block only. The towers do not
share weights; their outputs are L2-normalised and combined by a scaled dot
product. Early stopping uses mean slate NDCG@k on the chronological tuning
slice, with k set to the maximum number of clicks per impression in the training
slates; the final model is then refit on all training rows for the selected
epoch count. Both `"bpr"` and `"bce"` use the sampled negatives for each
positive. `"bpr"` scores the positive against each sampled negative; `"bce"`
scores the positive and each sampled negative with 1/0 labels.

In [13]:
def tune_ndcg(model):
    model.eval()
    scores = np.empty(len(TUNE_CLICKED), dtype=np.float32)
    with torch.no_grad():
        for start in range(0, len(TUNE_CLICKED), 200_000):
            end = min(start + 200_000, len(TUNE_CLICKED))
            users = torch.as_tensor(
                TRAIN_USER_FEATURES[TUNE_PROFILE_ROWS[start:end]],
                dtype=torch.float32,
                device=DEVICE,
            )
            items = torch.as_tensor(
                CONTENT_INPUTS[TUNE_ITEM_ROWS[start:end]], device=DEVICE
            )
            scores[start:end] = model(users, items).cpu().numpy()
    _, ndcg = mean_slate_metrics(TUNE_UIDS, TUNE_CLICKED, scores, TUNE_NDCG_K)
    return ndcg


In [14]:
def train_epoch(
    model, optimizer, profile_rows, pos_item_rows, neg_matrix, neg_len, rng, objective
):
    model.train()
    order = rng.permutation(len(profile_rows))
    total = 0.0
    for start in range(0, len(order), BATCH_SIZE):
        batch = order[start : start + BATCH_SIZE]
        users = torch.as_tensor(
            TRAIN_USER_FEATURES[profile_rows[batch]],
            dtype=torch.float32,
            device=DEVICE,
        )
        positives = torch.as_tensor(
            CONTENT_INPUTS[pos_item_rows[batch]], device=DEVICE
        )
        neg_rows = neg_matrix[batch]
        neg_lens = torch.as_tensor(neg_len[batch], device=DEVICE)
        neg_mask = (
            torch.arange(neg_rows.shape[1], device=DEVICE)[None, :]
            < neg_lens[:, None]
        )
        users_expanded = users[:, None, :].expand(
            -1, neg_rows.shape[1], -1
        ).reshape(-1, users.shape[1])
        negative_inputs = torch.as_tensor(
            CONTENT_INPUTS[neg_rows.reshape(-1)], device=DEVICE
        )
        optimizer.zero_grad(set_to_none=True)
        positive_scores = model(users, positives)
        negative_scores = model(users_expanded, negative_inputs).reshape(
            len(batch), neg_rows.shape[1]
        )
        if objective == "bpr":
            pair_loss = F.logsigmoid(
                positive_scores[:, None] - negative_scores
            )
            loss = -(pair_loss * neg_mask).sum() / neg_mask.sum()
        else:
            scores = torch.cat([positive_scores, negative_scores[neg_mask]])
            labels = torch.cat(
                [
                    torch.ones_like(positive_scores),
                    torch.zeros_like(negative_scores[neg_mask]),
                ]
            )
            loss = F.binary_cross_entropy_with_logits(scores, labels)
        loss.backward()
        optimizer.step()
        total += float(loss.item()) * len(batch)
    return total / len(order)


In [15]:
def build_model():
    return TwoTower().to(DEVICE)


In [16]:
TRAINED_LOSSES = {}
for objective in RUN_LOSSES:
    config = LOSS_CONFIGS[objective]
    artifact_dir = config["artifact_dir"]
    print(f"[{objective}] Training {config['model_name']}")

    model = build_model()
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    rng = np.random.default_rng(SEED + 1)
    training_log = []
    stopper = EarlyStopper(patience=PATIENCE)

    for epoch in range(1, NUM_EPOCHS + 1):
        started = time.time()
        profile_rows, pos_rows, neg_matrix, neg_len = sample_epoch(
            epoch - 1, fit_only=True
        )
        loss = train_epoch(
            model,
            optimizer,
            profile_rows,
            pos_rows,
            neg_matrix,
            neg_len,
            rng,
            objective,
        )
        ndcg = tune_ndcg(model)
        elapsed = time.time() - started
        training_log.append(
            {
                "phase": "tune",
                "epoch": epoch,
                "train_loss": loss,
                "tune_ndcg": ndcg,
                "ndcg_k": TUNE_NDCG_K,
                "seconds": elapsed,
            }
        )
        print(
            f"epoch {epoch:02d} loss={loss:.6f} "
            f"tune_ndcg@{TUNE_NDCG_K}={ndcg:.5f} ({elapsed:.1f}s)"
        )
        if stopper.step(ndcg, model, epoch):
            print("Early stopping")
            break

    stopper.restore(model)
    print(
        f"Best epoch: {stopper.best_epoch} "
        f"(tune_ndcg@{TUNE_NDCG_K}={stopper.best_score:.5f})"
    )

    final_model = build_model()
    optimizer = torch.optim.AdamW(
        final_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    rng = np.random.default_rng(SEED + 2)

    for epoch in range(1, stopper.best_epoch + 1):
        started = time.time()
        profile_rows, pos_rows, neg_matrix, neg_len = sample_epoch(
            epoch - 1, fit_only=False
        )
        loss = train_epoch(
            final_model,
            optimizer,
            profile_rows,
            pos_rows,
            neg_matrix,
            neg_len,
            rng,
            objective,
        )
        elapsed = time.time() - started
        training_log.append(
            {
                "phase": "refit",
                "epoch": epoch,
                "train_loss": loss,
                "tune_ndcg": float("nan"),
                "ndcg_k": TUNE_NDCG_K,
                "seconds": elapsed,
            }
        )
        print(f"refit epoch {epoch:02d} loss={loss:.6f} ({elapsed:.1f}s)")

    torch.save(
        {
            "state_dict": final_model.state_dict(),
            "config": {
                "user_input_dim": FEATURE_DIM,
                "item_input_dim": FEATURE_DIM,
                "hidden_dim": HIDDEN_DIM,
                "embed_dim": EMBED_DIM,
                "dropout": DROPOUT,
                "item_features": "text64 + entity100 + availability flags",
                "objective": objective,
                "epochs": stopper.best_epoch,
                "tune_ndcg": stopper.best_score,
                "ndcg_k": TUNE_NDCG_K,
                "batch_size": BATCH_SIZE,
                "learning_rate": LEARNING_RATE,
                "weight_decay": WEIGHT_DECAY,
                "seed": SEED,
                "negative_sampling": "per-epoch exposed negatives",
                "negative_count": int(CFG.negative_count),
                "sampler_epoch_base": 0,
                "training_data": "labels_train full training pool",
            },
        },
        artifact_dir / "model.pt",
    )
    pl.DataFrame(training_log).write_csv(artifact_dir / "training_log.csv")
    print("Saved model:", artifact_dir / "model.pt")

    TRAINED_LOSSES[objective] = {"model": final_model, "config": config}


[bpr] Training content_two_tower_mlp_bpr


C:\Users\Yong Rui\AppData\Local\Temp\ipykernel_5340\926966843.py:71: DeprecationWarning: In Polars 2.0, the default behavior for `empty_as_null` will change to `False`. To keep the current behavior, explicitly set `empty_as_null=True`.
  .explode("negative_news_codes")


epoch 01 loss=0.677759 tune_ndcg@31=0.42394 (10.1s)


C:\Users\Yong Rui\AppData\Local\Temp\ipykernel_5340\926966843.py:71: DeprecationWarning: In Polars 2.0, the default behavior for `empty_as_null` will change to `False`. To keep the current behavior, explicitly set `empty_as_null=True`.
  .explode("negative_news_codes")


epoch 02 loss=0.636764 tune_ndcg@31=0.43178 (3.8s)


C:\Users\Yong Rui\AppData\Local\Temp\ipykernel_5340\926966843.py:71: DeprecationWarning: In Polars 2.0, the default behavior for `empty_as_null` will change to `False`. To keep the current behavior, explicitly set `empty_as_null=True`.
  .explode("negative_news_codes")


... (89 outputs left out)

C:\Users\Yong Rui\AppData\Local\Temp\ipykernel_5340\926966843.py:71: DeprecationWarning: In Polars 2.0, the default behavior for `empty_as_null` will change to `False`. To keep the current behavior, explicitly set `empty_as_null=True`.
  .explode("negative_news_codes")


refit epoch 20 loss=0.530216 (3.2s)


C:\Users\Yong Rui\AppData\Local\Temp\ipykernel_5340\926966843.py:71: DeprecationWarning: In Polars 2.0, the default behavior for `empty_as_null` will change to `False`. To keep the current behavior, explicitly set `empty_as_null=True`.
  .explode("negative_news_codes")


refit epoch 21 loss=0.528614 (3.2s)
Saved model: c:\Users\Yong Rui\BT4222-Group-6-\5_Models\5_01_Candidate_Generation\artifacts\two_tower_content_bpr\model.pt


## 5. Results

### Candidate helpers

These helpers keep the candidate format, eligibility policy, history exclusion
and popularity tie-breaking consistent with the other candidate generators.

In [17]:
def popularity_select(order, n_eligible, excluded, k):
    excluded_set = set(map(int, excluded))
    selected = []
    for position in order:
        position = int(position)
        if position >= n_eligible or position in excluded_set:
            continue
        selected.append(position)
        if len(selected) == k:
            return np.asarray(selected, dtype=np.int64)
    raise ValueError(f"only {len(selected)} eligible articles; cannot select {k}")


def popularity_fill(available, popularity, news_code, count):
    pool = np.flatnonzero(available)
    order = np.lexsort((news_code[pool], -popularity[pool]))
    return pool[order[:count]]


def select_top_k(scores, popularity, news_code, k, excluded, fallback_order):
    if fallback_order is not None:
        return popularity_select(fallback_order, len(scores), excluded, k)

    available = np.ones(len(scores), dtype=bool)
    if len(excluded):
        available[excluded] = False
        scores = scores.copy()
        scores[excluded] = -np.inf

    finite = np.isfinite(scores)
    n_finite = int(np.count_nonzero(finite))
    if n_finite == 0:
        return popularity_fill(available, popularity, news_code, k)

    take = min(k, n_finite)
    if take < n_finite:
        threshold = scores[np.argpartition(-scores, take - 1)[:take]].min()
    else:
        threshold = scores[finite].min()

    above = np.flatnonzero(scores > threshold)
    need = take - above.size
    ties = np.flatnonzero(scores == threshold)
    if need > 0:
        if ties.size > need:
            ties = ties[np.lexsort((news_code[ties], -popularity[ties]))[:need]]
        chosen = np.r_[above, ties]
    else:
        chosen = above

    order = np.lexsort((news_code[chosen], -popularity[chosen], -scores[chosen]))
    selected = chosen[order].astype(np.int64)
    if selected.size < k:
        remaining = available.copy()
        remaining[selected] = False
        selected = np.r_[
            selected,
            popularity_fill(remaining, popularity, news_code, k - selected.size),
        ]
    return selected


In [18]:
def encode_batches(model, features, encode, batch=100_000):
    out = np.empty((len(features), EMBED_DIM), dtype=np.float32)
    model.eval()
    with torch.no_grad():
        for start in range(0, len(features), batch):
            values = torch.as_tensor(
                features[start : start + batch],
                dtype=torch.float32,
                device=DEVICE,
            )
            out[start : start + len(values)] = encode(values).cpu().numpy()
    return out


### Score the catalogue and write Top-1500 candidates

Item embeddings are static across the split because the item tower uses only
intrinsic content. The embeddings are sliced to the eligibility prefix, and
the tie-break and fallback use the timestamp's rolling prior-hour click order.
Each timestamp batch computes a user-by-candidate score matrix. Articles in
the request's full causal history are excluded; empty-profile requests use the
rolling prior-hour click order. Ties are broken by popularity, then
`news_code`.

In [19]:
for objective in RUN_LOSSES:
    state = TRAINED_LOSSES[objective]
    config = state["config"]
    model = state["model"]
    print(f"[{objective}] Scoring {config['model_name']}")

    target_profiles = load_profiles(TARGET_SPLIT)
    TARGET_USER_FEATURES, _ = load_profile_cache(TARGET_SPLIT)

    model.eval()
    user_emb = encode_batches(model, TARGET_USER_FEATURES, model.encode_users)
    item_emb = encode_batches(model, CONTENT_INPUTS_SORTED, model.encode_items)

    has_profile = (
        (TARGET_USER_FEATURES[:, -2] > 0) | (TARGET_USER_FEATURES[:, -1] > 0)
    )

    impression_uids = target_profiles["impression_uid"].to_numpy()
    timestamps = target_profiles["timestamp"].to_numpy().astype("datetime64[ns]")
    target_hours = hour_index_of(timestamps.astype(np.int64))
    history_rows = target_profiles["full_history_rows"].to_list()
    starts = np.flatnonzero(np.r_[True, timestamps[1:] != timestamps[:-1]])
    ends = np.r_[starts[1:], len(timestamps)]
    batches = list(zip(starts, ends))
    print(f"Scoring requests: {len(timestamps):,}")

    cutoff_values = np.arange(1, CANDIDATE_CAP + 1, dtype=np.int16)
    writer = CandidateWriter(config["recommendations_path"])
    current_hour = None
    popularity_sorted = None
    fallback_order = None
    started = time.time()
    try:
        for batch_no, (start, end) in enumerate(batches, 1):
            timestamp = timestamps[start]
            batch_hour = int(target_hours[start])
            if batch_hour != current_hour:
                popularity_sorted = hourly_popularity(batch_hour, ARTICLE_ROWS_SORTED)
                fallback_order = np.lexsort((NEWS_SORTED, -popularity_sorted))
                current_hour = batch_hour

            n_eligible = int(
                np.searchsorted(AVAILABLE_SORTED, timestamp, side="right")
            )
            scores = user_emb[start:end] @ item_emb[:n_eligible].T
            popularity_batch = popularity_sorted[:n_eligible]
            news_batch = NEWS_SORTED[:n_eligible]
            selected_batch = np.empty(
                (end - start, CANDIDATE_CAP), dtype=np.int64
            )
            for local, profile_row in enumerate(range(start, end)):
                rows = history_rows[profile_row]
                excluded = (
                    POS_IN_SORTED[np.asarray(rows, dtype=np.int64)]
                    if rows
                    else np.empty(0, dtype=np.int64)
                )
                fallback = None if has_profile[profile_row] else fallback_order
                selected_batch[local] = select_top_k(
                    scores[local],
                    popularity_batch,
                    news_batch,
                    CANDIDATE_CAP,
                    excluded,
                    fallback,
                )

            writer.add(
                np.repeat(impression_uids[start:end], CANDIDATE_CAP),
                np.tile(cutoff_values, end - start),
                news_batch[selected_batch.reshape(-1)],
            )
            if batch_no % 1000 == 0 or batch_no == len(batches):
                print(
                    f"Scored {batch_no:,}/{len(batches):,} timestamp batches; "
                    f"{writer.rows:,} rows",
                    flush=True,
                )
    finally:
        writer.close()

    elapsed = time.time() - started
    print(
        f"Wrote {len(timestamps) * CANDIDATE_CAP:,} candidate rows "
        f"in {elapsed:.1f}s"
    )
    print("Candidates:", config["recommendations_path"])
    state["recommendations_path"] = config["recommendations_path"]


[bpr] Scoring content_two_tower_mlp_bpr
Built profile cache for validation: 31,624 requests
Scoring requests: 31,624


C:\Users\Yong Rui\AppData\Local\Temp\ipykernel_5340\1536394434.py:6: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at B:\src\torch\torch\csrc\utils\tensor_numpy.cpp:219.)
  values = torch.as_tensor(


Scored 1,000/24,580 timestamp batches; 785,250 rows
Scored 2,000/24,580 timestamp batches; 1,648,500 rows
Scored 3,000/24,580 timestamp batches; 2,577,000 rows
Scored 4,000/24,580 timestamp batches; 3,550,500 rows
Scored 5,000/24,580 timestamp batches; 4,572,000 rows
Scored 6,000/24,580 timestamp batches; 5,601,000 rows
Scored 7,000/24,580 timestamp batches; 6,606,000 rows
Scored 8,000/24,580 timestamp batches; 7,643,250 rows
Scored 9,000/24,580 timestamp batches; 8,656,500 rows
Scored 10,000/24,580 timestamp batches; 9,698,250 rows
Scored 11,000/24,580 timestamp batches; 10,782,000 rows
Scored 12,000/24,580 timestamp batches; 11,851,500 rows
Scored 13,000/24,580 timestamp batches; 12,930,750 rows
Scored 14,000/24,580 timestamp batches; 13,996,500 rows
Scored 15,000/24,580 timestamp batches; 15,053,250 rows
Scored 16,000/24,580 timestamp batches; 16,095,000 rows
Scored 17,000/24,580 timestamp batches; 17,125,500 rows
Scored 18,000/24,580 timestamp batches; 18,094,500 rows
Scored 19,000

### Evaluation and comparison

Uses the shared `CandidateEvaluator`, which validates impression coverage,
cutoffs, uniqueness and time eligibility.

In [20]:
from cg_evaluation import CandidateEvaluator

for loss in RUN_LOSSES:
    state = TRAINED_LOSSES[loss]
    config = state["config"]
    evaluator = CandidateEvaluator(root=ROOT, split=TARGET_SPLIT)
    evaluation = evaluator.evaluate_to_csv(
        state["recommendations_path"],
        model_name=config["model_name"],
        m_values=M_VALUES,
        output_dir=RESULTS,
    )
    print(f"[{loss}]")
    print(evaluation.sort("m"))
    print("Results:", config["result_path"])


[bpr]
shape: (120, 7)
┌────────────────────┬────────────┬─────┬───────────────────┬───────────┬────────────┬─────────────┐
│ model              ┆ split      ┆ m   ┆ metric            ┆ value     ┆ numerator  ┆ denominator │
│ ---                ┆ ---        ┆ --- ┆ ---               ┆ ---       ┆ ---        ┆ ---         │
│ str                ┆ str        ┆ i64 ┆ str               ┆ f64       ┆ f64        ┆ f64         │
╞════════════════════╪════════════╪═════╪═══════════════════╪═══════════╪════════════╪═════════════╡
│ content_two_tower_ ┆ validation ┆ 50  ┆ catalogue_coverag ┆ 0.214523  ┆ 10198.0    ┆ 47538.0     │
│ mlp_bpr            ┆            ┆     ┆ e                 ┆           ┆            ┆             │
│ content_two_tower_ ┆ validation ┆ 50  ┆ cold_article_reca ┆ 0.002924  ┆ 1.0        ┆ 342.0       │
│ mlp_bpr            ┆            ┆     ┆ ll                ┆           ┆            ┆             │
│ content_two_tower_ ┆ validation ┆ 50  ┆ hit_rate          ┆ 0.03835

In [21]:
comparison_path = COMPARISON_PATH
if comparison_path.is_file():
    model_names = [LOSS_CONFIGS[loss]["model_name"] for loss in RUN_LOSSES]
    comparison = pl.read_csv(comparison_path).filter(
        pl.col("model").is_in(model_names)
    )
    loss_table = (
        comparison.filter(
            pl.col("metric").is_in(
                [
                    "recall_pooled",
                    "hit_rate",
                    "catalogue_coverage",
                    "personalisation",
                    "long_tail_recall",
                    "cold_article_recall",
                ]
            )
        )
        .pivot(on="model", index=["metric", "m"], values="value")
        .sort(["metric", "m"])
    )
    print(loss_table)
else:
    print("No comparison table yet; run the evaluation above first.")


shape: (90, 3)
┌────────────────────┬─────┬───────────────────────────┐
│ metric             ┆ m   ┆ content_two_tower_mlp_bpr │
│ ---                ┆ --- ┆ ---                       │
│ str                ┆ i64 ┆ f64                       │
╞════════════════════╪═════╪═══════════════════════════╡
│ catalogue_coverage ┆ 50  ┆ 0.214523                  │
│ catalogue_coverage ┆ 100 ┆ 0.308721                  │
│ catalogue_coverage ┆ 150 ┆ 0.377445                  │
│ catalogue_coverage ┆ 200 ┆ 0.430771                  │
│ catalogue_coverage ┆ 250 ┆ 0.474147                  │
│ …                  ┆ …   ┆ …                         │
│ recall_pooled      ┆ 550 ┆ 0.096787                  │
│ recall_pooled      ┆ 600 ┆ 0.101437                  │
│ recall_pooled      ┆ 650 ┆ 0.105482                  │
│ recall_pooled      ┆ 700 ┆ 0.109422                  │
│ recall_pooled      ┆ 750 ┆ 0.113154                  │
└────────────────────┴─────┴───────────────────────────┘


## Notes and limitations

- The user representation is specific to each request. Repeat users do not
  share a single latest profile; earlier requests use only earlier feedback.
- This is a pure content model: the item tower sees only text, entity and
  availability features. There are no user-ID, article-ID or popularity inputs.
- Excluding the causal history removes repeat-click opportunities, matching the
  content baseline. Recall therefore does not measure repeat clicks.
- Popularity is used only for tie-breaking and cold or short-list fallback. A
  request in hour `h` uses only completed hours before `h`, so no target-period
  or same-hour information enters the score.
- Training uses a chronological tuning slice, early stopping and a final refit
  on all training rows.
- The score is a ranking score. Grouped exposed negatives make the training
  positive rate much higher than held-out CTR, so the output is not calibrated.
- `available_from` is the project's observable-availability proxy, not a
  verified publication timestamp.